In [ ]:
import pandas as pd
import networkx as nx
import networkx as nx
from networkx.algorithms import community
from google.colab import drive
import random


In [ ]:
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/proj/hero-network.csv')

Mounted at /content/drive


## Data Access and Loading

The dataset is stored in Google Drive. I had problems loading the data. To solve that the data was accessed by mounting the drive within the notebook environment. The file `hero-network.csv` contains hero-to-hero interactions, where each row represents a connection between two characters.

The dataset is loaded into a pandas DataFrame.

In [ ]:
df_clean = df[df['hero1'] != df['hero2']]

## Data Cleaning

Before constructing the network, self-loops are removed from the dataset. A self-loop occurs when a hero is connected to themselves, which does not represent a meaningful interaction in this context.

Removing these ensures that all edges in the graph represent interactions between different heroes.

In [ ]:
G = nx.from_pandas_edgelist(df_clean, 'hero1', 'hero2')
print(f"Graph successfully built with {G.number_of_nodes()} heroes and {G.number_of_edges()} interactions.")

Graph successfully built with 6426 heroes and 167207 interactions.


## Graph Construction

The cleaned dataset is used to construct a network graph in which nodes represent heroes and edges represent interactions between them. The graph is built from the edge list, directly translating each pair of connected heroes into a link.

The resulting network contains **6,426 heroes (nodes)** and **167,207 interactions (edges)**, indicating a large and highly connected graph.

In [ ]:
degree_cent = nx.degree_centrality(G)
pagerank = nx.pagerank(G)
betweenness = nx.betweenness_centrality(G, k=1000)
centrality_df = pd.DataFrame({
    'Degree': degree_cent,
    'PageRank': pagerank,
    'Betweenness': betweenness
})
print(centrality_df.sort_values(by='PageRank', ascending=False).head(10))


                        Degree  PageRank  Betweenness
SPIDER-MAN/PETER PAR  0.270350  0.005323     0.074239
CAPTAIN AMERICA       0.296654  0.005117     0.056083
IRON MAN/TONY STARK   0.236887  0.004109     0.038117
WOLVERINE/LOGAN       0.213385  0.003870     0.033822
THING/BENJAMIN J. GR  0.220389  0.003674     0.024696
MR. FANTASTIC/REED R  0.214630  0.003571     0.025104
HUMAN TORCH/JOHNNY S  0.211829  0.003500     0.021523
SCARLET WITCH/WANDA   0.206226  0.003213     0.016441
THOR/DR. DONALD BLAK  0.200623  0.003200     0.022185
BEAST/HENRY &HANK& P  0.197198  0.003185     0.022447


## Centrality Analysis

To identify the most important and influential heroes in the network, three centrality measures are computed:

- **Degree Centrality** measures how many direct connections a hero has.
- **PageRank** captures overall influence by considering both the number and importance of connections.
- **Betweenness Centrality** identifies heroes that act as bridges between different parts of the network.

Since betweenness centrality was computationally so expensive, an approximate method is used by sampling *k = 1000*.

## Results Interpretation

The results show that **Spider-Man, Captain America, and Iron Man** rank among the most influential heroes across multiple centrality measures.

- **Spider-Man** has a high PageRank and betweenness centrality, indicating both strong influence and a key role in connecting different parts of the network.
- **Captain America** has the highest degree centrality, suggesting he has the most direct connections.
- **Iron Man and Wolverine** also rank highly, reinforcing their importance in the Marvel network.

In [ ]:
communities = list(community.greedy_modularity_communities(G))
print(f"Number of communities detected: {len(communities)}")
for i, comm in enumerate(communities[:5]):
    print(f"\nCommunity {i+1} (Size: {len(comm)}):")
    print(list(comm)[:20])

Number of communities detected: 67

Community 1 (Size: 2357):
['MAGDALENE/MARISSA DA', '3-D MAN/CHARLES CHAN', 'RAMOS, ROGER', 'MASTERMIND II/BOWERS', 'SEEKER III', "STARDUST/T'URIN G'AR", 'NOVA II/FRANKIE RAYE', 'ROMANOV, MICKEY', 'SCRIPPS, LAHOYA', 'JENNINGS, JOSEPH', 'LARGO, CHIEF', 'OBLIVION', 'NAUDA', 'BOWMAN, MAJ.', 'DEVOS THE DEVESTATOR', 'SZARDOS, MARGALI', 'GOLDEN ARCHER II/WYA', 'FREDD', 'CROSS, GENEVIEVE', 'GLORIOLE']

Community 2 (Size: 1726):
['FEROCIA/', 'MASTER OM', 'YRIK', 'LEEDS, NED', 'RINGLEADER', 'BLOOD-TIDE', 'LOPEZ, MARIA', 'I.Q./ISHMAEL QUESTOR', 'FORTUNATO, GIACOMO J', 'NIGHTWATCH/', 'VINCENT, DANNY', 'RICHMOND, DELROY', 'SCREAM', 'VULTURE II/BLACKIE D', 'HAVERSHAW, ANDREA', 'HELIUM', 'SABOTEUR/', 'RESTON, CLIVE', 'BAMBI', 'PILGRIM']

Community 3 (Size: 1372):
['MERCURY III/', 'MONTOYA, MARIA', 'GREENSONG', 'WATERS, SETH', 'KATANA/', 'CYCLOPS 2013', 'FIFOLET', 'LEGAULT', 'OSAMA', 'MANACLE/', 'BIRDBRAIN', 'GAMBLE, GRACIE', 'JONES, TIMOTHY', 'HALLOWEEN JACK', 'HUM

## Community Detection

Community detection is used to identify groups of heroes that are more strongly connected to each other than to the rest of the network. The greedy modularity algorithm is applied because it is a common method for detecting clusters in large networks.

The detection of 67 communities indicates that the Marvel network is highly modular.

The distribution of community sizes is highly uneven. A few very large communities (e.g., 2,357 and 1,726 nodes) dominate the network, while many smaller communities exist suggesting a core-periphery structure, where a central group of highly connected heroes forms the core, and smaller, more isolated groups exist on the edges.

The largest communities likely represent the main Marvel universe, where major characters frequently interact across multiple storylines. In contrast, smaller communities appear to correspond to more specialized groups, such as side characters, specific story arcs, or alternate universes.

Community 5 mainly contains heroes labeled “Mutant X”, indicating they belong to their own community proving that the algorithm correctly groups characters based on shared storylines.


In [ ]:

all_heroes = list(G.nodes())
random.seed(42)
random_heroes = random.sample(all_heroes, 5)

print("\nPredicting potential new collaborations for 5 random heroes:")

for hero_name in random_heroes:
    print(f"\n--- Random Sample: {hero_name} ---")

    potential_partners = [n for n in G.nodes() if n != hero_name and not G.has_edge(hero_name, n)]

    preds = nx.jaccard_coefficient(G, [(hero_name, n) for n in potential_partners])

    top_preds = sorted(list(preds), key=lambda x: x[2], reverse=True)[:5]
    for u, v, p in top_preds:
        print(f"  Predicted partner: {v} (Similarity Score: {p:.4f})")




Predicting potential new collaborations for 5 random heroes:

--- Random Sample: CONTRARES, ANASTASIA ---
  Predicted partner: EXILE (Similarity Score: 0.3077)
  Predicted partner: ACROBAT/CARL ZANTE (Similarity Score: 0.2667)
  Predicted partner: KURRGO (Similarity Score: 0.2667)
  Predicted partner: CASEY (Similarity Score: 0.2500)
  Predicted partner: ICONOCLAST (Similarity Score: 0.2500)

--- Random Sample: GELLER, URI ---
  Predicted partner: PINKWATER, ELLA (Similarity Score: 0.2500)
  Predicted partner: ELLINGTON, DR./TRAVI (Similarity Score: 0.2500)
  Predicted partner: COPPERHEAD/LAWRENCE  (Similarity Score: 0.2500)
  Predicted partner: SUNTURION II/MIKE ST (Similarity Score: 0.2500)
  Predicted partner: SCOPE (Similarity Score: 0.2500)

--- Random Sample: MOREAU, PHILLIP ---
  Predicted partner: HOLOCAUST (Similarity Score: 0.3007)
  Predicted partner: CHENEY, LILA (Similarity Score: 0.2759)
  Predicted partner: MILAN (Similarity Score: 0.2615)
  Predicted partner: HOUND (Si

## Link Prediction

Link prediction is used to estimate which heroes may be likely to interact in the future. For this analysis, five random heroes are selected from the graph, and the Jaccard coefficient is used to rank possible new connections.

The Jaccard coefficient compares shared neighbors between two heroes. A higher score means the two heroes have more similar connection patterns, making a future interaction more likely based on the network structure.

The results show the top predicted partners for each sampled hero based on shared neighbors. Higher similarity scores indicate stronger overlap in connections.

For example, **CONTRARES, ANASTASIA** has relatively high similarity with **EXILE (0.3077)**, while **DRAKE, MALCOLM** shows equal similarity scores with multiple heroes (0.3043), suggesting similar connection patterns.

In [25]:
!pip freeze > requirements.txt
from google.colab import files
files.download('requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
!python --version

Python 3.12.13
